# 03 – Environment-Test
Gymnasium-Environment mit einem Random-Agent testen.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
from src.data.loader import load_config, load_stations, get_coordinates
from src.api.google_maps import build_haversine_matrix
from src.environment.maintenance_env import MaintenanceEnv

config = load_config('../configs/config.yaml')
config['data']['raw_path'] = '../data/raw/charging_stations_wue.csv'

df = load_stations(config)
coords = get_coordinates(df, config)

# Kleinere Teilmenge für schnellen Test
subset = coords[:31]  # Depot + 30 Stationen
dur_matrix, _ = build_haversine_matrix(subset)

env = MaintenanceEnv(dur_matrix, config=config, render_mode='ansi')
print('Observation Space:', env.observation_space)
print('Action Space:', env.action_space)

In [ ]:
obs, info = env.reset(seed=42)
print('Initial Info:', info)

total_reward = 0.0
done = False

while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    done = terminated or truncated

print(f'\nEpisode beendet nach {info["step"]} Schritten')
print(f'Besucht: {info["n_visited"]}/{info["n_to_visit"]}')
print(f'Gesamt-Reward: {total_reward:.2f}')

## Gymnasium-Kompatibilität prüfen

In [ ]:
from gymnasium.utils.env_checker import check_env
check_env(env, warn=True)
print('check_env bestanden!')